# M4 — Escalation risk

Predicts whether a case will escalate. Class-balanced logistic regression on summary + category + channel + month + tenure. **Tags are excluded** (leakage). **Chronological 70/15/15 split**; the decision threshold is tuned on the validation set to a **minimum-recall = 0.80** target (missing an escalation is worse than a false alarm).

In [15]:
import sys, os, warnings; warnings.filterwarnings("ignore")
search = os.getcwd()
while search != os.path.dirname(search) and not os.path.exists(os.path.join(search, "case_pipeline.py")):
    search = os.path.dirname(search)
sys.path.insert(0, search)
import numpy as np, pandas as pd
import case_pipeline as P   # shared loading / splits / features / evaluation

In [16]:
df = P.load_clean()
df["escalated"] = df["escalated"].astype(bool)

# Inbound features only. category is predicted upstream by M2 at serve time.
FEATURES   = ["case_summary", "category", "channel", "month_num", "customer_tenure_months"]
TARGET     = "escalated"
POSITIVE   = True
MIN_RECALL = 0.80
classes    = [False, True]

train, val, test = P.time_split(df)
steps = [P.t_word(), P.t_char(), P.t_cat(["category", "channel"]),
         P.t_num(["month_num", "customer_tenure_months"])]
model = P.make_clf(steps).fit(train[FEATURES], train[TARGET])
pos_col = list(model.classes_).index(POSITIVE)

# Tune the threshold on validation to reach the recall floor.
val_proba = model.predict_proba(val[FEATURES])[:, pos_col]
threshold = P.recall_targeted_threshold((val[TARGET] == POSITIVE), val_proba, min_recall=MIN_RECALL)

In [ ]:
test_proba = model.predict_proba(test[FEATURES])[:, pos_col]
pred = np.where(test_proba >= threshold, True, False)

baseline = (~df[TARGET]).mean()                 # accuracy of always predicting "no escalation"
baseline_pr_auc = (test[TARGET] == POSITIVE).mean()   # no-skill PR-AUC = positive prevalence on the test set
metrics  = P.evaluate(test[TARGET], pred, classes, proba=test_proba, positive=POSITIVE, baseline=baseline)
metrics["baseline_PR_AUC"] = round(float(baseline_pr_auc), 3)
metrics.update(P.binary_at_threshold(test[TARGET], test_proba, POSITIVE, threshold))
metrics["min_recall_target"] = MIN_RECALL
metrics

In [ ]:
heldout = test[["case_id", "case_summary", TARGET]].copy().reset_index(drop=True)
heldout["prob"] = test_proba.round(3)
heldout["predicted"] = pred
heldout["confidence"] = heldout["prob"]

deployed = P.make_clf(steps).fit(df[FEATURES], df[TARGET])
P.save_bundle("m4_escalation", {
    "model": deployed, "target": TARGET, "features": FEATURES, "classes": classes, "kind": "binary",
    "label": "Escalation risk", "threshold": round(float(threshold), 3), "positive": POSITIVE,
    "metrics": metrics, "test": heldout,
    "report": P.class_report(test[TARGET], pred, classes),
    "confusion": P.conf_matrix(test[TARGET], pred, classes),
    "meta": {"contributor": "E", "approach": "escalation risk, recall-targeted threshold",
             "split": "time 70/15/15", "threshold_rule": "min recall 0.80 on validation",
             "inputs": "summary + category + channel + month + tenure (no tags — leakage)"}})